In [12]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)

True

In [2]:
import duckdb

DB_PATH = "./data/ab_events.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)
event_log = con.execute("SELECT * FROM events").df()
con.close()

print(event_log.shape)
event_log.head()

(2024484, 7)


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN


In [3]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    df = con.execute("SHOW tables").df()

print(df)

                     name
0                  events
1    fct_ab_buckets_daily
2  int_ab_events_bucketed
3           stg_event_log


In [4]:
%%sql

SELECT * FROM event_log


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN
...,...,...,...,...,...,...,...
2024479,2025-02-28 23:19:03,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024480,2025-02-28 23:19:44,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024481,2025-02-28 23:20:04,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
2024482,2025-02-28 23:21:08,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN


In [5]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    fct_ab_buckets_daily = con.execute("SELECT * FROM fct_ab_buckets_daily").df()


In [6]:
%%sql buckets <<

SELECT * FROM fct_ab_buckets_daily


,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
0,2025-01-01,US,172,num01,a,67,55,9,2,1,12,12,8,2,1,26.14
1,2025-01-01,US,161,num01,b,49,42,7,0,0,9,9,7,0,0,0.00
2,2025-01-01,US,154,num01,b,59,52,5,1,1,12,12,4,1,1,141.77
3,2025-01-01,GB,63,num01,a,29,22,6,1,0,8,8,6,1,0,0.00
4,2025-01-01,GB,127,num01,a,15,13,1,0,1,2,2,1,0,1,58.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69197,2025-02-25,DE,192,num01,a,9,8,1,0,0,1,1,1,0,0,0.00
69198,2025-02-25,DE,121,num01,a,2,2,0,0,0,1,1,0,0,0,0.00
69199,2025-02-25,DE,71,num01,b,6,5,1,0,0,2,2,1,0,0,0.00
69200,2025-02-25,DE,156,num01,a,5,4,0,0,1,2,2,0,0,1,58.34


In [7]:
buckets[
    (buckets.date_day=="2025-01-10")
    & (buckets.country=='US')
]

,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
1009,2025-01-10,US,29,num01,a,23,20,3,0,0,5,5,3,0,0,0.00
1011,2025-01-10,US,125,num01,b,68,58,8,2,0,9,9,6,2,0,0.00
1012,2025-01-10,US,137,num01,a,75,57,11,3,4,16,16,11,3,4,141.29
1013,2025-01-10,US,198,num01,b,82,66,10,4,2,14,14,10,4,2,108.03
1017,2025-01-10,US,179,num01,a,34,26,6,1,1,7,7,6,1,1,41.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61710,2025-01-10,US,17,num01,b,26,23,3,0,0,4,4,3,0,0,0.00
61711,2025-01-10,US,127,num01,b,38,35,3,0,0,8,8,3,0,0,0.00
61718,2025-01-10,US,91,num01,a,51,42,8,1,0,9,9,7,1,0,0.00
61729,2025-01-10,US,47,num01,a,49,37,9,2,1,9,9,7,2,1,36.09


In [8]:
(
    buckets.groupby('date_day')
    .total_unique_users.sum()
    .to_frame()
    .tail(20)
)
    

,total_unique_users
date_day,
2025-02-09,6580
2025-02-10,6471
2025-02-11,5958
2025-02-12,6242
2025-02-13,6667
2025-02-14,6517
2025-02-15,7371
2025-02-16,7180
2025-02-17,6293


# запуск экспериментов

In [14]:
from database import load_buckets
from analytics import run, significant_results, display_tt

buckets = load_buckets("num01")
results = run(buckets)
display_tt(significant_results(results))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,total_events,a,b,1 013 721,1 010 763,196 971,192 857,1.690,3.63,1.723,0.289,3.092
1,purchase_amount,a,b,697 835,789 274,196 971,192 857,14.974,7.06,7.558,8.221,21.728
2,watch_count,a,b,131 193,138 505,196 971,192 857,7.775,16.51,1.755,6.317,9.234
3,u_watch,a,b,117 874,123 515,196 971,192 857,7.017,20.88,1.311,5.977,8.058
4,add_to_cart_count,a,b,22 648,24 054,196 971,192 857,8.284,7.39,4.220,4.800,11.768
5,u_add_to_cart,a,b,22 141,23 468,196 971,192 857,8.045,7.47,4.090,4.701,11.389
6,purchase_count,a,b,12 716,14 387,196 971,192 857,14.868,9.57,5.641,9.910,19.827
7,u_purchase,a,b,12 510,14 101,196 971,192 857,14.426,9.58,5.563,9.621,19.231


In [15]:
buckets["country"].value_counts().head(10)

country
US    23594
GB    23100
DE    22508
Name: count, dtype: int64

In [16]:
buckets_de = load_buckets("num01", country="DE")
results_de = run(buckets_de)
display_tt(significant_results(results_de))

,metric,control,treatment,metric_control,metric_treatment,userday_control,userday_treatment,effect_size_pct,tstat_obs,monitoring_mde_pct,CI_low_effect,CI_high_effect
0,purchase_amount,a,b,115 088,138 229,37 405,38 724,16.750,3.49,17.536,1.231,32.269
1,watch_count,a,b,23 997,26 671,37 405,38 724,7.274,6.54,4.094,3.840,10.708
2,u_watch,a,b,21 544,23 888,37 405,38 724,7.097,8.25,3.243,4.437,9.756
3,add_to_cart_count,a,b,3 910,4 351,37 405,38 724,7.634,2.73,10.780,-1.038,16.306
4,u_add_to_cart,a,b,3 829,4 243,37 405,38 724,7.206,2.64,10.470,-1.246,15.658
5,purchase_count,a,b,2 203,2 741,37 405,38 724,20.210,5.20,13.799,7.519,32.901
6,u_purchase,a,b,2 165,2 690,37 405,38 724,20.067,5.23,13.852,7.526,32.607
